<a href="https://colab.research.google.com/github/Mru321/CSI-Cross-Environment-Generalization/blob/main/CNN%20%2B%20LSTM%20model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Mount drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os

RAW_DIR = "/content/drive/MyDrive/Project/Project1/raw_data/renew_dataset/tvt_open_source/raw_traces"
PROCESSED_DIR = "/content/drive/MyDrive/Project/Project1/processed"

os.makedirs(PROCESSED_DIR, exist_ok=True)

In [ ]:
import os
import numpy as np

files = sorted([f for f in os.listdir(PROCESSED_DIR) if f.endswith(".npy")])

file_info = []

for f in files:
    path = os.path.join(PROCESSED_DIR, f)

    label = 0 if "NLOS" in f else 1

    # FIXED env extraction
    env = f.replace(".npy", "")

    file_info.append((path, label, env))

In [ ]:
train_envs = [
    "channe14_LOS_cluster1",
    "channe14_LOS_cluster2",
    "channe14_NLOS_cluster1",
    "channe14_NLOS_cluster2",
    "channe14_NLOS_cluster3"
]

test_envs = [
    "channe14_LOS_cluster3",
    "channe14_LOS_cluster4",
    "channe14_NLOS_cluster4",
    "channe14_NLOS_cluster5"
]

In [ ]:
for path, label, env in file_info:
    print(f"File: {os.path.basename(path)}")
    print(f"  Label: {label} ({'LOS' if label==1 else 'NLOS'})")
    print(f"  Env: {env}")
    print("-"*40)

File: channe14_LOS_cluster1.npy
  Label: 1 (LOS)
  Env: channe14_LOS_cluster1
----------------------------------------
File: channe14_LOS_cluster2.npy
  Label: 1 (LOS)
  Env: channe14_LOS_cluster2
----------------------------------------
File: channe14_LOS_cluster3.npy
  Label: 1 (LOS)
  Env: channe14_LOS_cluster3
----------------------------------------
File: channe14_LOS_cluster4.npy
  Label: 1 (LOS)
  Env: channe14_LOS_cluster4
----------------------------------------
File: channe14_NLOS_cluster1.npy
  Label: 0 (NLOS)
  Env: channe14_NLOS_cluster1
----------------------------------------
File: channe14_NLOS_cluster2.npy
  Label: 0 (NLOS)
  Env: channe14_NLOS_cluster2
----------------------------------------
File: channe14_NLOS_cluster3.npy
  Label: 0 (NLOS)
  Env: channe14_NLOS_cluster3
----------------------------------------
File: channe14_NLOS_cluster4.npy
  Label: 0 (NLOS)
  Env: channe14_NLOS_cluster4
----------------------------------------
File: channe14_NLOS_cluster5.npy
  L

In [ ]:
def create_sequences(data, window_size=50, stride=10):
    sequences = []

    n = len(data)

    for i in range(0, n - window_size + 1, stride):
        sequences.append(data[i:i+window_size])

    return np.array(sequences)

In [ ]:
X_train, y_train = [], []
X_test, y_test = [], []

for path, label, env in file_info:

    data = np.load(path)
    seqs = create_sequences(data, window_size=50, stride=10)

    if env in train_envs:
        X_train.append(seqs)
        y_train.append(np.full(len(seqs), label))

    elif env in test_envs:
        X_test.append(seqs)
        y_test.append(np.full(len(seqs), label))

In [ ]:
for _, _, env in file_info:
    print(env)

channe14_LOS_cluster1
channe14_LOS_cluster2
channe14_LOS_cluster3
channe14_LOS_cluster4
channe14_NLOS_cluster1
channe14_NLOS_cluster2
channe14_NLOS_cluster3
channe14_NLOS_cluster4
channe14_NLOS_cluster5


In [ ]:
for path, label, env in file_info:
    if env in train_envs:
        print(env, "→ TRAIN")
    elif env in test_envs:
        print(env, "→ TEST")
    else:
        print(env, "→ ❌ NOT MATCHED")

channe14_LOS_cluster1 → TRAIN
channe14_LOS_cluster2 → TRAIN
channe14_LOS_cluster3 → TEST
channe14_LOS_cluster4 → TEST
channe14_NLOS_cluster1 → TRAIN
channe14_NLOS_cluster2 → TRAIN
channe14_NLOS_cluster3 → TRAIN
channe14_NLOS_cluster4 → TEST
channe14_NLOS_cluster5 → TEST


In [ ]:
for path, label, env in file_info:
    data = np.load(path)
    print(env, "frames:", len(data))

channe14_LOS_cluster1 frames: 8140
channe14_LOS_cluster2 frames: 8159
channe14_LOS_cluster3 frames: 8112
channe14_LOS_cluster4 frames: 8101
channe14_NLOS_cluster1 frames: 6081
channe14_NLOS_cluster2 frames: 4054
channe14_NLOS_cluster3 frames: 4058
channe14_NLOS_cluster4 frames: 4057
channe14_NLOS_cluster5 frames: 4063


In [ ]:
X_train = np.concatenate(X_train, axis=0)
y_train = np.concatenate(y_train, axis=0)

X_test = np.concatenate(X_test, axis=0)
y_test = np.concatenate(y_test, axis=0)

print("Train:", X_train.shape)
print("Test:", X_test.shape)

Train: (3027, 50, 72)
Test: (2416, 50, 72)


In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, LSTM, Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping

In [ ]:
model = Sequential()

# 🔹 CNN Layer (spatial feature extraction)
model.add(Conv1D(filters=32, kernel_size=3, activation='relu',
                 input_shape=(50, 72)))
model.add(BatchNormalization())
model.add(MaxPooling1D(pool_size=2))

# 🔹 Optional deeper CNN
model.add(Conv1D(filters=64, kernel_size=3, activation='relu'))
model.add(BatchNormalization())
model.add(MaxPooling1D(pool_size=2))

# 🔹 LSTM Layer (temporal learning)
model.add(LSTM(64))

# 🔹 Dense layers
model.add(Dense(32, activation='relu'))
model.add(Dropout(0.3))

# 🔹 Output
model.add(Dense(1, activation='sigmoid'))

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [ ]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d (Conv1D)                 │ (None, 48, 32)         │         6,944 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 48, 32)         │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ (None, 24, 32)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 22, 64)         │         6,208 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 22, 64)         │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_1 (MaxPooling1D)  │ (None, 11, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 64)             │        33,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 48,673 (190.13 KB)

 Trainable params: 48,481 (189.38 KB)

 Non-trainable params: 192 (768.00 B)

In [ ]:
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True
)

In [ ]:
history = model.fit(
    X_train,
    y_train,
    epochs=20,
    batch_size=32,
    validation_split=0.2,
    callbacks=[early_stop]
)

Epoch 1/20
76/76 ━━━━━━━━━━━━━━━━━━━━ 6s 23ms/step - accuracy: 0.9798 - loss: 0.1025 - val_accuracy: 0.9422 - val_loss: 0.1487
Epoch 2/20
76/76 ━━━━━━━━━━━━━━━━━━━━ 1s 17ms/step - accuracy: 1.0000 - loss: 0.0040 - val_accuracy: 0.9868 - val_loss: 0.0279
Epoch 3/20
76/76 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step - accuracy: 0.9959 - loss: 0.0126 - val_accuracy: 1.0000 - val_loss: 0.0018
Epoch 4/20
76/76 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step - accuracy: 0.9996 - loss: 0.0022 - val_accuracy: 1.0000 - val_loss: 0.0011
Epoch 5/20
76/76 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step - accuracy: 0.9934 - loss: 0.0156 - val_accuracy: 1.0000 - val_loss: 0.0046
Epoch 6/20
76/76 ━━━━━━━━━━━━━━━━━━━━ 2s 28ms/step - accuracy: 1.0000 - loss: 8.5271e-04 - val_accuracy: 1.0000 - val_loss: 0.0012
Epoch 7/20
76/76 ━━━━━━━━━━━━━━━━━━━━ 2s 29ms/step - accuracy: 1.0000 - loss: 5.0000e-04 - val_accuracy: 1.0000 - val_loss: 0.0014


In [ ]:
loss, acc = model.evaluate(X_test, y_test)

print("Test Accuracy:", acc)

76/76 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.8969 - loss: 0.3448
Test Accuracy: 0.8969370722770691


In [ ]:
y_pred = model.predict(X_test)
y_pred = (y_pred > 0.5).astype(int)

76/76 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step


In [ ]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.76      1.00      0.87       803
           1       1.00      0.85      0.92      1613

    accuracy                           0.90      2416
   macro avg       0.88      0.92      0.89      2416
weighted avg       0.92      0.90      0.90      2416

